# BEAMM Example with Simulated Data

This notebook demonstrates how to use the BEAMM package with simulated data. It includes steps for importing the necessary modules, loading the data, and running the BEAMM algorithm to make predictions.

## Importing Required Modules
You will need to import the following modules from the BEAMM package:

```python
from beamm import BEAMMMapper, import_data, generate_metrics, mean_standardized_log_loss
```

### Note:
 - If you want to use physical data, you will have to normalize the data and provide the normalization parameters to the BEAMMMapper class. The normalization parameters can be obtained from the training data.

## Configuration

The `config.toml` file defines the BEAMM parameters to run the magnetic prediction. You can change these values based on your data. It will need to be changed depending on the area that you are mapping. Differences between building magnetic fields and coverage magnetic fields out in a field can be vastly different due to varying ferromagnetic materials in the ground.


In [ ]:
from beamm import BEAMMMapper, import_data, generate_metrics, mean_standardized_log_loss
import torch
import matplotlib.pyplot as plt

torch.cuda.empty_cache()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
torch.manual_seed(42)

X_train, Y_train = import_data(
    "../beamm-dataset/large-scale-magnetic-mapping/simu1D2D3D/simu1D_training.csv", device=device
)
X_test, Y_test = import_data("../beamm-dataset/large-scale-magnetic-mapping/simu1D2D3D/simu_test.csv", device=device)

## Running the BEAMM Mapper for an Offline Train and Predict

Running a single step for training is fairly straight forward. You simply run the `update_and_predict` method in `BEAMMMapper` and that will train and predict based on the test data given to it. Re-initializing the class will reset the sequential Monte Carlo, giving the behavior of a single step train and predict.

If you want to generate metrics like RMSE, MSLL and coverage you can use the `generate_metrics` function.

In [ ]:
offline_mapper = BEAMMMapper(device=device)

mean_offline, std_offline, _ = offline_mapper.update_and_predict(
    xd_all=X_train,
    yd_all=Y_train,
    xd_new=torch.empty((0, 3), device=device),
    yd_new=torch.empty((0, 3), device=device),
    xtd=X_test,
)

metrics_offline = generate_metrics(
    mean=mean_offline, sigma_2=std_offline**2, y_test=Y_test.cpu().numpy(), y_train=Y_train.cpu().numpy()
)

print(X_train.shape, Y_train.shape, X_test.shape, Y_test.shape)
# Plotting the results

plt.figure(figsize=(12, 6))
plt.title("Mean Predictions")
plt.plot(X_test.detach().cpu().numpy()[:, 0], Y_test.detach().cpu().numpy()[:, 0], label="True Values", alpha=0.5)
plt.plot(X_test.detach().cpu().numpy()[:, 0], mean_offline[:, 0], label="Predicted Mean", alpha=0.5)
plt.fill_between(
    X_test.detach().cpu().numpy()[:, 0],
    mean_offline[:, 0] - 2 * std_offline[:, 0],
    mean_offline[:, 0] + 2 * std_offline[:, 0],
    color="gray",
    alpha=0.2,
    label="Confidence Interval (2 std)",
)
plt.xlabel("Positioon")
plt.ylabel("Field Value $B_x$")
plt.legend()
plt.show()

## Continual Learning

If you want to run continual learning, we can test it out on a batched dataset given here. The functional changes are minimal as the `BEAMMMapper` handles the continual learning internally.

In [ ]:
X_train, Y_train = import_data("../beamm-dataset/curl_free/curl_free_train.csv", device=device)
X_test, Y_test = import_data("../beamm-dataset/curl_free/curl_free_test.csv", device=device)

n_batches = 5
batch_size = X_train.shape[0] // n_batches

continual_beamm = BEAMMMapper(device=device)
results = {"rmse": [], "msll": []}

for i in range(n_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if i < n_batches - 1 else X_train.shape[0]

    xd_batch = X_train[start_idx:end_idx]
    yd_batch = Y_train[start_idx:end_idx]
    x_acc = X_train[:end_idx]
    y_acc = Y_train[:end_idx]

    mean_continual, std_continual, _ = continual_beamm.update_and_predict(
        xd_all=x_acc,
        yd_all=y_acc,
        xd_new=xd_batch,
        yd_new=yd_batch,
        xtd=X_test,
    )
    metrics_continual = generate_metrics(
        mean=mean_continual, sigma_2=std_continual**2, y_test=Y_test.cpu().numpy(), y_train=Y_train.cpu().numpy()
    )
    results["rmse"].append(metrics_continual["rmse"])
    results["msll"].append(metrics_continual["msll"])

# Plotting the results for continual learning
plt.figure(figsize=(12, 6))
ax1 = plt.subplot(1, 2, 1)
ax1.set_title("RMSE over Batches")
ax1.plot(range(1, n_batches + 1), results["rmse"], marker="o")
ax1.set_xlabel("Batch Number")
ax1.set_ylabel("RMSE")
ax1.set_xticks(range(1, n_batches + 1))
ax1.set_xticklabels([f"Batch {i}" for i in range(1, n_batches + 1)])

ax2 = plt.subplot(1, 2, 2)
ax2.set_title("MSLL over Batches")
ax2.plot(range(1, n_batches + 1), results["msll"], marker="o", color="orange")
ax2.set_xlabel("Batch Number")
ax2.set_ylabel("MSLL")
ax2.set_xticks(range(1, n_batches + 1))
ax2.set_xticklabels([f"Batch {i}" for i in range(1, n_batches + 1)])
plt.tight_layout()
plt.show()